# Getting Data From Satellite in CSV

To use this code go to the last section (section 2) and change to the desired values

## 1. Function def and importing libraries

In [ ]:
import sys
sys.path.append("../common/")
from satelite import ColSatellite
from satelite import csv_from_sat
from satelite import statAnalisisData
import numpy as np
import gstools as gs
import matplotlib.pyplot as plt
import ee
import geemap
from datetime import datetime
from dateutil.relativedelta import relativedelta
import pandas as pd
import seaborn as sns



In [ ]:
ee.Authenticate()
ee.Initialize(
  project='ee-hides',
  opt_url='https://earthengine-highvolume.googleapis.com'
)

In [ ]:
def getAllDateData(satelite, layer, region, startDate, endDate, ceros):
    """Obtiene todos los datos de satélite para un satélite y capa específicos dentro de un rango de fechas.

    Esta función obtiene los datos de satélite para cada día dentro del rango de fechas especificado
    y los concatena en un solo DataFrame.

    Args:
        satelite (str): Nombre del satélite.
        layer (str): Nombre de la capa.
        startDate (str): Fecha de inicio en formato 'AAAA-MM-DD'.
        endDate (str): Fecha de fin en formato 'AAAA-MM-DD'.

    Returns:
        pandas.DataFrame: Los datos de satélite para todas las fechas en forma de pandas DataFrame.
    """

    date_format = '%Y-%m-%d'
    date_i = startDate

    s_date_i = date_i + 'T11:00:00Z' # Tener en cuenta que esto es UTC y cambia dependiendo la región
    e_date_i = date_i + 'T23:59:00Z'
    
    emissionSat = ColSatellite(satelliteName, layer) # creates a satellite object
    imageCollection = emissionSat.get_image_collection(s_date_i, e_date_i) # obtain the imgae collection
    emissionsData = emissionSat.getImageCollectionAsDF(imageCollection, region, ceros)
    try:
        while date_i != endDate:
            print('shape: ', emissionsData.shape, 'Fecha:', date_i)
            date_i = datetime.strptime(date_i, date_format)
            date_i = date_i + relativedelta(days=1)
            date_ii=date_i + relativedelta(days=1)
            date_i = str(date_i)
            date_i = date_i.split()[0]
            date_ii=str(date_ii)
            date_ii = date_ii.split()[0]
    
            s_date_i = date_i + 'T11:00:00Z'
            e_date_i = date_i + 'T23:59:00Z'
            try:
                imageCollection_i = emissionSat.get_image_collection(s_date_i, e_date_i) # obtain the imgae collection
                emissionsData_i = emissionSat.getImageCollectionAsDF(imageCollection_i, region, ceros) 
                emissionsData = pd.concat([emissionsData, emissionsData_i])
                print('shape: ', emissionsData_i.shape, 'longitud')
            except  Exception as e:
                print(e)
                print('no hay dato para la fecha: ', s_date_i)

        return emissionsData
    except KeyboardInterrupt:
        print('el script se detuvo, el resultado se guardó')
        return emissionsData
    else:
        print('falló')



## 2. Selecting the region of interest

In [ ]:
satelliteName = "COPERNICUS/S5P/OFFL/L3_CH4"
layer = 'CH4_column_volume_mixing_ratio_dry_air_bias_corrected'
methaneSat = ColSatellite(satelliteName, layer) # creates a satellite object
methaneSat.get_available_regions() # Muestra las regiones disponibles
fronterasMaritimasCol = methaneSat.get_roi('fronterasMaritimasCol') # selecciona la region que ya trae el objeto, la entrada es el ID

# Colombia = ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level2")#Datos de fronteras a nivel municipal
# Mun_col = Colombia.filter(ee.Filter.eq('ADM0_NAME', 'Colombia'))#Filtro de datos para Colombia

# Mun_santander_ant = Mun_col.filter(ee.Filter.inList('ADM1_NAME', ['Santander','Antioquia']))# Se selecciona el departamento de Boyacá

# municipios = ['Barrancabermeja', 'Yondo']
# municipios_metano = ['Barrancabermeja', 'Yondo', 'Puerto Wilches']
# municipios_geometry = Mun_santander_ant.filter(ee.Filter.inList('ADM2_NAME', municipios))# Se selecciona la región de interés
# municipios_geometry_metano = Mun_santander_ant.filter(ee.Filter.inList('ADM2_NAME', municipios_metano))# Se selecciona la región de interés


#Fronteras=ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level0")#Datos de fronteras nacionales
#Front_col=Fronteras.filter(ee.Filter.eq('ADM0_NAME', 'Colombia'))#Filtro de los anteriores datos para Colombia

Municipios= ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level1")#Datos de fronteras a nivel municipal
Mun_col=Municipios.filter(ee.Filter.eq('ADM0_NAME', 'Colombia'))#Filtro de datos para Colombia

N_dep = len(Mun_col.getInfo()['features'])

dep_list = []

for i in range(N_dep):
    dep_list.append(Mun_col.getInfo()['features'][i]['properties']['ADM1_NAME'])
    
dep_list
#roi = Front_col

In [ ]:
dep_list.pop(26)


In [ ]:
def partition_list(lst, n):
    # Verificar que n sea válido
    if n <= 0:
        raise ValueError("El número de particiones debe ser mayor que 0")
    if n > len(lst):
        raise ValueError("El número de particiones no puede ser mayor que el tamaño de la lista")
    
    # Calcular el tamaño de cada partición
    size = len(lst) // n
    remainder = len(lst) % n
    
    # Crear las particiones
    result = []
    start = 0
    
    for i in range(n):
        # Ajustar el tamaño si hay elementos restantes
        partition_size = size + (1 if i < remainder else 0)
        end = start + partition_size
        
        # Agregar la partición a la lista resultado
        result.append(lst[start:end])
        start = end
    
    return result

partitioned_dep_list = partition_list(dep_list,16)


partitioned_dep_list

roi1_list = partitioned_dep_list[0]
roi2_list = partitioned_dep_list[1]
roi3_list = partitioned_dep_list[2]
roi4_list = partitioned_dep_list[3]
roi5_list = partitioned_dep_list[4]
roi6_list = partitioned_dep_list[5]
roi7_list = partitioned_dep_list[6]
roi8_list = partitioned_dep_list[7]
roi9_list = partitioned_dep_list[8]
roi10_list = partitioned_dep_list[9]
roi11_list = partitioned_dep_list[10]
roi12_list = partitioned_dep_list[11]
roi13_list = partitioned_dep_list[12]
roi14_list = partitioned_dep_list[13]
roi15_list = partitioned_dep_list[14]
roi16_list = partitioned_dep_list[15]


roi1 = Mun_col.filter(ee.Filter.inList('ADM1_NAME', roi1_list))# Se selecciona la región de interés
roi2 = Mun_col.filter(ee.Filter.inList('ADM1_NAME', roi2_list))# Se selecciona la región de interés
roi3 = Mun_col.filter(ee.Filter.inList('ADM1_NAME', roi3_list))# Se selecciona la región de interés
roi4 = Mun_col.filter(ee.Filter.inList('ADM1_NAME', roi4_list))# Se selecciona la región de interés
roi5 = Mun_col.filter(ee.Filter.inList('ADM1_NAME', roi5_list))# Se selecciona la región de interés
roi6 = Mun_col.filter(ee.Filter.inList('ADM1_NAME', roi6_list))# Se selecciona la región de interés
roi7 = Mun_col.filter(ee.Filter.inList('ADM1_NAME', roi7_list))# Se selecciona la región de interés
roi8 = Mun_col.filter(ee.Filter.inList('ADM1_NAME', roi8_list))# Se selecciona la región de interés
roi9 = Mun_col.filter(ee.Filter.inList('ADM1_NAME', roi9_list))# Se selecciona la región de interés
roi10 = Mun_col.filter(ee.Filter.inList('ADM1_NAME', roi10_list))# Se selecciona la región de interés
roi11 = Mun_col.filter(ee.Filter.inList('ADM1_NAME', roi11_list))# Se selecciona la región de interés
roi12 = Mun_col.filter(ee.Filter.inList('ADM1_NAME', roi12_list))# Se selecciona la región de interés
roi13 = Mun_col.filter(ee.Filter.inList('ADM1_NAME', roi13_list))# Se selecciona la región de interés
roi14 = Mun_col.filter(ee.Filter.inList('ADM1_NAME', roi14_list))# Se selecciona la región de interés
roi15 = Mun_col.filter(ee.Filter.inList('ADM1_NAME', roi15_list))# Se selecciona la región de interés
roi16 = Mun_col.filter(ee.Filter.inList('ADM1_NAME', roi16_list))# Se selecciona la región de interés


all_roi = [roi1, roi2, roi3, roi4, roi5, roi6, roi7, roi8, roi9, roi10, roi11, roi12, roi13, roi14, roi15, roi16]

In [ ]:
roi = roi2
Map = geemap.Map(center=(4.6,-74),zoom=9)
Map.addLayer(roi2, {}, 'Colombia',True,0.4)
Map


In [ ]:
#roi2.getInfo()['features'][1]['properties']['ADM1_NAME']
partitioned_dep_list

## 3. Creating the satellite Object and getting values

In [ ]:
# Change the satellite name and layer to different emission
# satelliteName = "COPERNICUS/S5P/OFFL/L3_CH4"
# layer = 'CH4_column_volume_mixing_ratio_dry_air'
# satelliteNhttps://drive.google.com/file/d/1QmlX9pawkhz4SFae2ieNj0XKP8r5BB56/view?usp=sharingame = "MODIS/061/MCD19A2_GRANULES"
# layer = 'Optical_Depth_055'
# To obtain a desired date range change the values below
initDate = '2022-04-09' 
finalDate = '2024-11-02'


In [ ]:
# Using functions and creating csv files
ceros = True #verdadero para quitar los ceros
emissionsData_df = getAllDateData(satelliteName, layer, roi, initDate, finalDate, ceros)
emissionsData_df['date'] = emissionsData_df['time'].apply(lambda x: datetime.fromtimestamp(float(x)/ 1000))

if ceros:
    emissionsData_df.to_csv('csvResults/'+ '' + layer + '_' + initDate + '_' + finalDate + '_antioquia_corr' +'.csv', index=False)
else:
    emissionsData_df.to_csv('csvResults/'+ 'Zeros_' + layer + '_' + initDate + '_' + finalDate + '.csv', index=False)

In [ ]:
ceros = True #verdadero para quitar los ceros
count = 4
for roi_i in all_roi[4:]:
    region_name = '_'.join(partitioned_dep_list[count])
    print('getting data of'+ region_name + ' region')
    emissionsData_df = getAllDateData(satelliteName, layer, roi_i, initDate, finalDate, ceros)
    emissionsData_df['date'] = emissionsData_df['time'].apply(lambda x: datetime.fromtimestamp(float(x)/ 1000))
    finalDate_save = emissionsData_df['date'].iloc[-1].strftime("%Y-%m-%d")
    
    if ceros:
        emissionsData_df.to_csv('csvResults/'+ '' + layer + '_' + initDate + '_' + finalDate_save + region_name +'.csv', index=False)
        count += 1
    else:
        emissionsData_df.to_csv('csvResults/'+ 'Zeros_' + layer + '_' + initDate + '_' + finalDate + '.csv', index=False)

In [ ]:
emissionsData_df['date'].iloc[-1].strftime("%Y-%m-%d")

In [ ]:
aa = ['a', 'b']
'_'.join(aa)

In [ ]:
2019-02-12 12:27:57